# Piecewise Conversion

Real converters don't have a single efficiency. A condensing boiler is most efficient at mid-load, worse at low load (cycling/standby losses) and worse at high load (hotter flue). A heat pump's COP varies with ambient temperature.

You'll meet **`ConversionCurve`** — a piecewise-linear coupling between two or more flows. The optimizer picks an operating point on the curve at every timestep. We'll pair it with the `Status` you saw in T3 so the boiler can also turn off.

In [ ]:
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

from fluxopt import Carrier, ConversionCurve, Converter, Effect, Flow, Port, Status, optimize

pio.renderers.default = 'notebook_connected'

## 1. The curve

A 100 kW condensing boiler with three efficiency segments:

| Gas range (kW) | Slope (η = heat/gas) | Why |
|---|---|---|
| 0 – 30  | 0.75 | Cycling and standby losses dominate |
| 30 – 70 | 0.90 | Sweet spot — flue cool enough to condense |
| 70 – 100 | 0.67 | Flue gas hotter, less condensation |

Breakpoints: gas at `[0, 30, 70, 100]` paired with heat at `[0, 22.5, 58.5, 78.5]`.

In [ ]:
gas_bp = [0, 30, 70, 100]
heat_bp = [0, 22.5, 58.5, 78.5]
slopes = [(heat_bp[i + 1] - heat_bp[i]) / (gas_bp[i + 1] - gas_bp[i]) for i in range(3)]

fig = px.line(x=gas_bp, y=heat_bp, markers=True, height=320, labels={'x': 'gas input (kW)', 'y': 'heat output (kW)'})
for i, eta in enumerate(slopes):
    fig.add_annotation(
        x=(gas_bp[i] + gas_bp[i + 1]) / 2,
        y=(heat_bp[i] + heat_bp[i + 1]) / 2,
        text=f'η = {eta:.2f}',
        showarrow=False,
        yshift=-18,
        font={'color': '#16a34a', 'size': 12},
    )
fig.update_layout(template='plotly_white', margin={'l': 50, 'r': 20, 't': 10, 'b': 40})
fig

## 2. Solve

Two ingredients compose into the converter:

- The **curve** itself, in the dict form `{flow_short_id: breakpoints}` — every listed flow shares the same interpolation weights.
- A **`Status`** that gates the entire curve with one binary; when off, every curve flow is pinned to its first breakpoint (here `(0, 0)`).

A backup heat source at €0.20/kWh (4× the gas-derived cost) gives "off" a meaningful alternative — without it, the boiler would always run since gas is cheaper than backup at every load.

In [ ]:
n = 24
timesteps = [datetime(2024, 1, 15) + timedelta(hours=h) for h in range(n)]
demand = np.array(
    [
        10,
        10,
        15,
        20,
        25,
        35,
        50,
        65,
        70,
        60,
        50,
        45,
        40,
        35,
        30,
        25,
        20,
        30,
        55,
        70,
        78,
        75,
        60,
        35,
    ]
)

In [ ]:
result = optimize(
    timesteps=timesteps,
    carriers=[Carrier('gas'), Carrier('heat')],
    effects=[Effect('cost', unit='EUR')],
    ports=[
        Port('gas_grid', imports=[Flow('gas', size=500, effects_per_flow_hour={'cost': 0.05})]),
        Port('backup', imports=[Flow('heat', size=200, effects_per_flow_hour={'cost': 0.20})]),
        Port(
            'workshop', exports=[Flow('heat', size=float(demand.max()), fixed_relative_profile=demand / demand.max())]
        ),
    ],
    converters=[
        Converter(
            'boiler',
            inputs=[Flow('gas', short_id='fuel')],
            outputs=[Flow('heat', size=100)],
            conversion=ConversionCurve(
                {'fuel': gas_bp, 'heat': heat_bp},
                status=Status(min_uptime=2, effects_per_startup={'cost': 2.0}),
            ),
        ),
    ],
    objective_effects='cost',
)
print(f'Total cost: {result.objective:.2f} EUR')

## 3. Read the result

Two views:

- **Operating points** — each timestep's `(gas, heat)` overlaid on the curve. With the status binary in play, points cluster on the efficient segments and a few sit at the origin (boiler off, backup serves demand).
- **Time series** — boiler heat / backup heat / demand, plus the boiler's on/off binary.

The on/off variable is at `result.solution['component--on']` (component-level — `Status` lives on the curve, not on a single flow).

In [ ]:
gas_rate = result.flow_rate('boiler(fuel)').values
heat_rate = result.flow_rate('boiler(heat)').values
backup_rate = result.flow_rate('backup(heat)').values
on = result.solution['component--on'].sel(component='boiler').values

fig = go.Figure()
fig.add_trace(go.Scatter(x=gas_bp, y=heat_bp, mode='lines+markers', name='curve', line_color='#636efa'))
fig.add_trace(
    go.Scatter(
        x=gas_rate,
        y=heat_rate,
        mode='markers',
        name='operating points',
        marker={'symbol': 'x', 'size': 9, 'color': '#ef553b'},
    )
)
fig.update_layout(
    height=320,
    margin={'l': 50, 'r': 20, 't': 10, 'b': 40},
    template='plotly_white',
    xaxis_title='gas input (kW)',
    yaxis_title='heat output (kW)',
)
fig

In [ ]:
times = result.flow_rates.coords['time'].values
df = pd.concat(
    [
        pd.DataFrame({'time': times, 'value': heat_rate, 'panel': 'heat (kW)', 'series': 'boiler'}),
        pd.DataFrame({'time': times, 'value': backup_rate, 'panel': 'heat (kW)', 'series': 'backup'}),
        pd.DataFrame({'time': times, 'value': demand, 'panel': 'heat (kW)', 'series': 'demand'}),
        pd.DataFrame({'time': times, 'value': on, 'panel': 'boiler on/off', 'series': 'on'}),
    ],
    ignore_index=True,
)
fig = px.line(df, x='time', y='value', color='series', facet_row='panel', line_shape='hv', height=460)
fig.update_yaxes(matches=None, title='')
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.update_layout(template='plotly_white', margin={'l': 50, 'r': 20, 't': 30, 'b': 20})
fig

## Recap

`ConversionCurve` replaces a fixed efficiency with a piecewise-linear coupling. It composes with `Status` to give you the on/off binary at `solution['component--on']`. The dict form scales to **three or more flows** simply by adding entries — every flow shares the same interpolation weights, which is how a CHP with load-dependent heat-to-power ratio is modeled.

You've now seen the full vocabulary: `Carrier`, `Flow`, `Port`, `Converter`, `Effect`, `Storage`, `Status`, `ConversionCurve`. Two more to go: `Sizing` (capacity optimization) and `Investment` (build-timing).

**Next**: [Sizing](05-sizing.ipynb) — let the optimizer pick the boiler and tank capacities.